# Verify variants — filtered (`hvg5000`) vs non-filtered (`all_genes`)

A re-runnable check of the preprocessing outputs, so the gene-count flow and the
PCA-vs-scGPT inputs can be audited at any time. Covers:

1. Gene-count flow across `convert -> scgpt -> targets` (HVG filter, then scGPT-OOV drop).
2. **Why scGPT drops genes** (the `id_in_vocab` vocabulary mask).
3. Cell alignment across the three files (required for the PCA fix).
4. What `.X` holds (CPM vs log-normalized) and where `X_pca` / `X_scGPT` come from.
5. `hvg5000` vs `all_genes` gene-set comparison.
6. UMAPs: PCA baseline vs scGPT embedding, by cancer type and by paclitaxel viability.

See `docs/steps/02-preprocessing-and-embeddings.md` for the narrative.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt

DATA_ROOT = Path('/Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542')
VARIANTS = ['hvg5000', 'all_genes']
FILES = {
    'convert':    'SCP542_CCLE.h5ad',
    'embeddings': 'SCP542_CCLE_scGPT_human_embeddings.h5ad',
    'targets':    'SCP542_CCLE_scGPT_human_embeddings_with_targets.h5ad',
}

def load(variant, stage, backed='r'):
    return sc.read_h5ad(DATA_ROOT / variant / FILES[stage], backed=backed)

## 1. Gene-count flow per variant

`convert` applies the HVG filter; `scgpt` then drops out-of-vocabulary (OOV) genes.

In [ ]:
rows = []
for v in VARIANTS:
    g = {s: load(v, s).n_vars for s in FILES}
    rows.append({
        'variant': v,
        'convert (HVG)': g['convert'],
        'embeddings (post-OOV)': g['embeddings'],
        'targets': g['targets'],
        'OOV dropped': g['convert'] - g['embeddings'],
    })
pd.DataFrame(rows).set_index('variant')

## 2. Why does scGPT drop genes?

scGPT has a **fixed gene vocabulary** from pretraining. `gen_embeds.py` tags each gene with
`id_in_vocab` (>= 0 if known, -1 if not) and keeps only the known ones. Genes not in the
vocabulary cannot be tokenized, so they are dropped before embedding. The dropped genes are
exactly the OOV count above.

In [ ]:
for v in VARIANTS:
    emb = load(v, 'embeddings')
    if 'id_in_vocab' in emb.var.columns:
        iv = emb.var['id_in_vocab'].astype(int)
        print(f'{v}: kept (id_in_vocab>=0) = {(iv >= 0).sum()},  OOV (== -1) = {(iv < 0).sum()}')
    else:
        print(f'{v}: embeddings file already subset to in-vocab genes -> n_vars = {emb.n_vars}')

## 3. Cell alignment across files

The PCA fix computes `X_pca` from the `convert` counts and stores it in the `targets` file, so
the cells must be in the same order in both. This must print `True` for every variant.

In [ ]:
for v in VARIANTS:
    a = load(v, 'convert'); b = load(v, 'embeddings'); c = load(v, 'targets')
    ok = np.array_equal(a.obs_names, b.obs_names) and np.array_equal(b.obs_names, c.obs_names)
    print(f'{v}: convert == embeddings == targets cell order -> {ok}  (n_cells = {c.n_obs})')

## 4. What `.X` holds, and where `X_pca` / `X_scGPT` come from

After the fix, the `targets` `.X` stays **CPM** (large row sums), `X_scGPT` is 512-d (from the
in-vocab genes), and `X_pca` is 50-d computed on the **full convert gene set** (HVG-5000 or all).

In [ ]:
for v in VARIANTS:
    t = load(v, 'targets', backed=None)
    X = t.X
    Xd = X[:200].toarray() if sp.issparse(X) else np.asarray(X[:200])
    rowsum = float(Xd.sum(1).mean())
    kind = 'CPM (raw-ish)' if Xd.max() > 100 else 'log-normalized'
    print(f'{v}:')
    print(f'  .X genes={t.n_vars}  max={Xd.max():.1f}  mean row-sum={rowsum:,.0f}  -> looks {kind}')
    print(f'  X_scGPT={t.obsm["X_scGPT"].shape}  X_pca={t.obsm["X_pca"].shape}')
    print(f'  convert gene count (what X_pca should be built from) = {load(v, "convert").n_vars}')

## 5. `hvg5000` vs `all_genes` gene sets

How many of the 5,000 HVGs survive into the all-genes embedding, and the OOV fractions.

In [ ]:
hvg_conv = set(load('hvg5000', 'convert').var_names)
hvg_emb  = set(load('hvg5000', 'embeddings').var_names)
all_emb  = set(load('all_genes', 'embeddings').var_names)
all_conv = set(load('all_genes', 'convert').var_names)
print(f'HVG-5000 selected genes:                 {len(hvg_conv)}')
print(f'  of those in scGPT vocab (embedded):    {len(hvg_emb)}  ({len(hvg_conv - hvg_emb)} OOV)')
print(f'  of those also present in all_genes emb:{len(hvg_emb & all_emb)}')
print(f'Full transcriptome genes:                {len(all_conv)}')
print(f'  of those in scGPT vocab (embedded):    {len(all_emb)}  ({len(all_conv - all_emb)} OOV)')

## 6. UMAPs — PCA baseline vs scGPT embedding

For each variant: UMAP of `X_pca` (left) and `X_scGPT` (right), colored by cancer type and by
paclitaxel viability. PCA is expected to form discrete tissue 'islands'; scGPT a continuous
shared manifold. **Compute-heavy** (neighbors + UMAP on ~53k cells, four times).

In [ ]:
def umap_for_rep(adata, rep, seed=42):
    a = adata.copy()
    sc.pp.neighbors(a, use_rep=rep, random_state=seed)
    sc.tl.umap(a, random_state=seed)
    return a.obsm['X_umap']

VARIANT_TO_PLOT = 'hvg5000'  # switch to 'all_genes' to inspect the full-transcriptome variant
adata = load(VARIANT_TO_PLOT, 'targets', backed=None)
umaps = {rep: umap_for_rep(adata, rep) for rep in ['X_pca', 'X_scGPT']}
print('done:', {k: v.shape for k, v in umaps.items()})

In [ ]:
color_cat = adata.obs['Cancer_type'].astype('category')
color_via = pd.to_numeric(adata.obs['viability_paclitaxel'], errors='coerce')

fig, axes = plt.subplots(2, 2, figsize=(15, 13))
for j, rep in enumerate(['X_pca', 'X_scGPT']):
    U = umaps[rep]
    # row 0: cancer type
    for code, name in enumerate(color_cat.cat.categories):
        m = color_cat.values == name
        axes[0, j].scatter(U[m, 0], U[m, 1], s=2, alpha=0.5, label=name)
    axes[0, j].set_title(f'{rep} — Cancer_type'); axes[0, j].set_xticks([]); axes[0, j].set_yticks([])
    # row 1: paclitaxel viability
    sc_ = axes[1, j].scatter(U[:, 0], U[:, 1], s=2, c=color_via.values, cmap='viridis')
    axes[1, j].set_title(f'{rep} — viability_paclitaxel'); axes[1, j].set_xticks([]); axes[1, j].set_yticks([])
    fig.colorbar(sc_, ax=axes[1, j], shrink=0.7)
axes[0, 1].legend(markerscale=3, fontsize=6, ncol=2, loc='center left', bbox_to_anchor=(1.0, 0.5))
fig.suptitle(f'{VARIANT_TO_PLOT}: PCA vs scGPT UMAP', fontsize=14)
plt.tight_layout(); plt.show()